# Chapter 13 — Memory Optimization

Part 5: Layouts and data movement.

**Ladder:** naive PyTorch strided reference (Method 01) → a coalesced Triton
copy/scale kernel (Method 02) → a fused, vectorized kernel that owns four
elements per thread and never reloads its input (Method 03).

**Study tip.** Fix one 1-D tile and count the bytes and the cache lines by hand
before reading any kernel. For $N=1024$ `float32`:

$$
Q_r=Nb=1024\cdot4=4096\ \text{B},\qquad
Q_w=Nb=4096\ \text{B},\qquad
Q=Q_r+Q_w=8192\ \text{B}=8\ \text{KiB}.
$$

A $128$-byte line holds $128/4=32$ elements, so the copy touches $32$ lines per
direction and $64$ lines in total. A $256$-element tile is $8$ lines. Write
those three numbers on paper; every optimization in this chapter changes one of
them.

**Thread.**

- **Recurring idea.** A kernel is judged by the bytes it moves and the lines it
  touches, not by the arithmetic it performs.
- **Chain.** Coalescing → stride → alignment → vector width → bandwidth → cache
  → registers → shared memory → access patterns → redundant loads → redundant
  stores → fusion.
- **Promise.** Each section states one rule, one byte/line example, one Triton
  habit, and one failure mode; every code cell asserts the arithmetic.

**Learning objectives**

- Explain why coalescing turns 32 lane addresses into one or four $128$-byte
  transactions, and count the lines for an arbitrary stride.
- Compute bytes moved, arithmetic intensity, and roofline time for a 1-D kernel
  before running it.
- Align a base pointer and a tile, emit $16$-byte vector loads/stores, and say
  how many elements each thread owns.
- Reason about cache reuse, register pressure, and shared-memory staging as
  traffic and occupancy budgets.
- Remove redundant loads and stores by fusing an affine chain, and prove the
  traffic reduction with arithmetic.

[Index](../../00_index.ipynb) · [← Previous](../ch12_layouts/12_complete.ipynb) · [Next →](../../part6_reductions/ch14_reductions/14_complete.ipynb)

**Estimated time:** 80–95 min · **Difficulty:** Intermediate

**Prerequisites:** Chapter 12 (layouts, elements per thread, `num_warps`);
Chapter 6 (pointer arithmetic and masks); Chapter 7 (why elementwise kernels are
memory-bound).

**Book apparatus:** [Setup](../../docs/SETUP.md) · [Notation](../../docs/NOTATION.md) ·
[Glossary](../../docs/GLOSSARY.md) · [API index](../../docs/API_INDEX.md) ·
[Formula index](../../docs/FORMULA_INDEX.md) · [Diagrams](../../docs/DIAGRAMS.md) ·
[Troubleshooting](../../docs/TROUBLESHOOTING.md) ·
[Exercise hints](../../docs/SOLUTIONS.md)

## Concept diagrams

### Contiguous versus strided access

```text
contiguous, stride 1 element (4 B), one warp = 32 lanes
lane :  0    1    2    3   ...   31
addr : +0   +4   +8  +12   ... +124
       [---------- one 128-byte line ----------]

stride 4 elements (16 B), one warp = 32 lanes
lane :  0    1    2    3   ...   31
addr : +0  +16  +32  +48   ... +496
       [ line 0 ][ line 1 ][ line 2 ][ line 3 ]
       four lines, only 32 B of each line used
```

### One cache line

```text
128-byte line = 4 sectors x 32 bytes = 32 float32 words

+---------+---------+---------+---------+
| 0 .. 7  | 8 ..15  |16 ..23  |24 ..31  |
+---------+---------+---------+---------+
  sector 0  sector 1  sector 2  sector 3
```

The hardware moves a line (or a sector) even when one lane asks for one word.
That is why the *line count*, not the useful-byte count, sets the traffic.

### Vectorized load and store

```text
scalar path:  4 x ld.global.b32    -> 4 instructions, 4 address calculations
vector path:  1 x ld.global.v4.b32 -> 1 instruction,  16 bytes
              [ x0 x1 x2 x3 ]          one 16-byte vector
```

A store is the mirror image: `st.global.v4.b32` writes 16 bytes at once.

### Redundant load elimination

```text
two passes (materialized intermediate):
  load x -> *2 -> store t -> load t -> +1 -> store y
  traffic = 4 vectors = 16N bytes

one fused pass:
  load x -> *2 + 1 -> store y
  traffic = 2 vectors = 8N bytes
```

The second form is not a different mathematical function. It is the same
function with the intermediate kept in a register instead of HBM.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if (root / "00_Basic").exists():
    root = root / "00_Basic"
while root.name != "00_Basic" and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root))

import torch
import triton
import triton.language as tl
from ttools import DEVICE, HAS_GPU, require_gpu, run_if_gpu

print("device:", DEVICE, "| GPU:", HAS_GPU)

## Part I — Mathematics and concepts

**Thread.** Memory optimization is traffic accounting.

- **Quantities.** $N$ elements, $b$ bytes each, line $L=128$ B, tile $B$,
  threads $T=32W$.
- **Bounds.** $Q=Q_{\text{read}}+Q_{\text{write}}$; $t\ge Q/\mathrm{BW}$;
  $\mathrm{AI}=\text{FLOPs}/Q$.
- **Test.** A kernel is memory-bound when $\mathrm{AI}$ is below the machine
  balance; every section changes $Q$, the transaction count, or the hierarchy
  level.
- **Plan.** The worked example after the chapter map fixes $N=1024$; each code
  cell asserts its own numbers.

### 13.1 Coalesced access

**Connection.** Chapter 12 fixed the lane map; Chapter 7 showed elementwise
kernels are memory-bound. Coalescing is the first property of that map.

**Rule.** Lane $\ell$ touches $a_0+\ell b$. A warp is coalesced when those
addresses are consecutive.

**Formula.** $T_{\text{line}}=\lceil 32b/L\rceil$ with $L=128$ B; for
`float32`, $T_{\text{line}}=1$.

**Example.** 32 `float32` lanes span $32\cdot4=128$ B, one line. A $B=256$
tile is $256\cdot4/128=8$ lines. Traffic stays $Q=2Nb$ when every fetched byte
is used.

**Where.** Dense copies, scales, and epilogues; the coalescer merges lane
requests before L2, so a warp on one line issues one transaction instead of 32.

**Triton.** `offs = pid * BLOCK + tl.arange(0, BLOCK)`; keep the fast axis on
the lane index; `tl.max_contiguous` and `tl.multiple_of` mark the run.

**Failure.** `row * cols + col` with `col` fixed; a scattered mask can split one
request into many.

**Bridge.** A stride $s\gt 1$ turns this one line into several.

### 13.2 Uncoalesced access

**Connection.** Builds on 13.1: same warp and same line count, but the lane
addresses now step by $s$ elements.

**Rule.** Lane $\ell$ touches $a_0+\ell sb$; the warp spans $32sb$ bytes.

**Formula.** $T_{\text{line}}(s)=\lceil 32sb/L\rceil$.

**Example.** For `float32`, $s=4$ spans $512$ B, or $4$ lines, with only
$128$ B useful ($4\times$ overfetch); $s=32$ spans $4096$ B, or $32$ lines
($1/32$ utilization). A $256$-element tile at $s=4$ spans $4096$ B, or $32$
lines, against $8$ contiguous lines.

**Where.** Row-major column reads, gathers, unfused transposes, `x[::s]`; the
arithmetic is unchanged, only the address map is bad. If the layout cannot
change, accept the traffic and measure it.

**Triton.** Reorder offsets so the lane index is unit-stride; load a 2-D tile;
transpose the logical tile before the store.

**Failure.** `offs = base + tl.arange(0, BLOCK) * stride`, or a 2-D kernel whose
fast axis is the row.

**Bridge.** Even a unit-stride lane map needs natural alignment before it can
issue a wide access.

### 13.3 Memory alignment

**Connection.** Follows 13.2: the lane map may be unit-stride, but a $16$-byte
access also needs its address to be a multiple of $16$.

**Rule.** A $w$-byte access is aligned when $a\bmod w=0$.

**Formula.** For a `float32` tile, $a(p,t)=a_{\text{base}}+4(pB+t)$; a
$16$-byte vector requires $a\bmod16=0$.

**Example.** With $a_{\text{base}}=4096$ and $B=256$, tile starts are
$4096+1024p$, divisible by $16$; vector groups $t=0,4,8,\ldots$ start aligned,
while $t=1,2,3$ sit at $4,8,12$ bytes into a vector. A base offset of $+12$ B
is $4$-byte aligned but not $16$-byte aligned.

**Where.** Vector instructions require natural alignment; an unaligned
$16$-byte access may issue as two or four narrower accesses, raising the
instruction count and defeating the coalescer.

**Triton.** Power-of-two blocks; normal PyTorch allocation; `tl.multiple_of(offs,
4)` when the block start is a multiple of four elements.

**Failure.** Slicing at an odd element (`x[1:]`), a non-zero storage offset, or
pointer arithmetic that loses the base alignment.

**Bridge.** Once alignment is legal, vector width decides how many elements one
instruction moves.

### 13.4 Vectorized loads

**Connection.** Builds on 13.3: alignment makes a wide load legal; vector width
sets how many elements it carries.

**Rule.** The hardware width is $w=128$ bits $=16$ bytes, so $v=w/b$ elements
per vector.

**Formula.** $v=4$ for `float32`, $8$ for `float16`/`bfloat16`, $2$ for
`float64`; a $B$-element tile needs $B/v$ vector loads, and each thread owns
$e/v$ vectors when $e=B/T$.

**Example.** $B=256$ `float32` gives $256/4=64$ vector loads; with $T=64$
threads that is one $16$-byte vector per thread, with $T=32$ it is two.

**Where.** A vector load is one instruction and one address calculation instead
of four; scalar loads can make a memory-bound kernel issue-bound before it
saturates HBM.

**Triton.** Contiguous, aligned offsets; choose $e$ as a multiple of $4$ for
`float32`; `tl.max_contiguous(tl.multiple_of(offs, 4), 4)`.

**Failure.** A masked tail forcing predicated scalars, a stride other than one,
or a false alignment hint.

**Bridge.** Stores use the same width and the same count.

### 13.5 Vectorized stores

**Connection.** Mirrors 13.4 on the write path: the same width, the same count,
and the same alignment requirement.

**Rule.** Stores use $v=128/b$ elements per $16$-byte vector, so a $B$-element
tile needs $B/v$ stores.

**Formula.** $N_{\text{store vectors}}=B/v$.

**Example.** A $256$-element `float32` tile needs $64$ `st.global.v4.b32`
instructions. A store miss may allocate the line before writing it, so a
partial-line store can cost an extra read. A kernel that writes one scalar per
lane can turn one line into 32 write transactions.

**Where.** Epilogues, transposed outputs, and any kernel that writes a
materialized intermediate.

**Triton.** Use the same contiguous offsets for load and store, mask both, and
write the final dtype once.

**Failure.** Storing through a strided output (`out[:, j]`), a masked tail with
a vector hint, or an intermediate the next kernel rereads.

**Bridge.** With the transaction count fixed, the next question is how many
bytes per second those transactions achieve.

### 13.6 Memory bandwidth

**Connection.** Builds on 13.5: having counted lines and vector stores, divide
total bytes by time to get the figure of merit.

**Rule.** $\mathrm{BW}=Q/t$, with $Q=Q_{\text{read}}+Q_{\text{write}}$; for a
copy, $Q=2Nb$.

**Formula.** $t\ge\max(N/P_{\text{peak}},Q/\mathrm{BW})$ and
$\mathrm{AI}=\text{FLOPs}/Q$.

**Example.** $N=2^{26}$ `float32` gives
$Q=2\cdot67{,}108{,}864\cdot4=536{,}870{,}912$ B $=512$ MiB; at $3.0$ TB/s
the roof is $0.179$ ms. The $N=1024$ copy moves $8$ KiB and needs $2.73$ ns at
that bandwidth, below launch overhead.

**Where.** Every memory-bound kernel; effective bandwidth above spec indicates
cache reuse, far below indicates uncoalesced access or spills.

**Triton.** Time many repetitions with CUDA events, use a working set larger
than L2, and compute $Q/t$.

**Failure.** Timing one launch, timing a cache-resident tensor, or counting
useful bytes while hardware moves whole lines.

**Bridge.** A measured bandwidth above the DRAM roof means cache reuse is
already happening.

### 13.7 Cache behavior

**Connection.** Explains the 13.6 anomaly: a run can beat the DRAM roof because
L2 serves repeated lines.

**Rule.** A $128$-byte line has four $32$-byte sectors; a working set $W$ is
reused when $W\le C_{\text{L2}}$ and the reuse distance is short.

**Formula.** $W\le C_{\text{L2}}\Rightarrow$ repeated reads can hit L2 instead
of HBM.

**Example.** A $1024\times1024$ `float32` matrix is $4$ MiB, or
$4\cdot1024\cdot1024/128=32{,}768$ lines; it fits a large L2, so a column-major
second pass can hit cache even though its per-warp line count is still bad.

**Where.** Tiled GEMM, normalization-row reuse, panel sweeps.

**Triton.** Order programs so a panel is consumed before eviction; use larger
tiles but check the shared-memory and register budgets; the schedule matters
more than a cache modifier.

**Failure.** Extrapolating from a cache-resident tensor, or assuming a store is
free because it "goes through the cache": a store still consumes bandwidth and
may allocate a line.

**Bridge.** Reuse must be held somewhere: the next level is the register file.

### 13.8 Register usage

**Connection.** Builds on 13.7: cache reuse reduces HBM traffic, but a reused
value must stay live in registers.

**Rule.** With $W$ warps and $T=32W$ threads, $e=B/(32W)$ elements per thread
per live tile.

**Formula.** $k$ live tiles need about $ke$ data registers per thread, before
addresses, masks, and temporaries.

**Example.** $B=1024$ and $W=4$ give $e=8$; two live tiles need $16$ data
registers per thread. A spill goes to local memory, which is HBM-backed.

**Where.** Registers are the fastest storage; between arithmetic operations
they are the only place a tile should live. Too many live tiles spill, while too
few leave insufficient work to hide latency.

**Triton.** Reduce `BLOCK`, split the kernel, or recompute a cheap value;
inspect compiled register and spill metadata.

**Failure.** "Bigger tile is always faster"; holding input, intermediate, and
output at once can spill and add traffic.

**Bridge.** When registers cannot hold the reuse, shared memory stages it.

### 13.9 Shared memory

**Connection.** Builds on 13.8: shared memory is the next staging level when a
reusable panel is too large for registers.

**Rule.** Triton stages `tl.dot`, layout conversions, and some reductions
through shared memory; the budget is $M_{\text{smem}}=s(B_MB_K+B_KB_N)b$.

**Formula.** $s$ is the number of pipeline stages; for $B_M=B_N=B_K=64$,
`float16`, and $s=3$, $M_{\text{smem}}=3(64\cdot64+64\cdot64)\cdot2=49{,}152$
B $=48$ KiB.

**Example.** A $100$ KiB per-SM budget fits two such blocks; a $48$ KiB block
is not free just because it is on chip.

**Where.** Panel reuse in GEMM and layout conversions; elementwise kernels need
none.

**Triton.** Tune `BLOCK_M`, `BLOCK_N`, `BLOCK_K`, and `num_stages` together.

**Failure.** Assuming shared memory is free, ignoring bank conflicts, or
leaving one resident block per SM; an accidental layout conversion in
elementwise code can also pay a shared-memory round trip.

**Bridge.** Resources only pay off when the lane map uses them; that is
§13.10.

In [ ]:
# Why: registers and shared memory are budgets; check them before choosing a tile.
BLOCK = 1024
W = 4
T = 32 * W
elements_per_thread = BLOCK // T
assert (T, elements_per_thread) == (128, 8)
assert 2 * elements_per_thread == 16

# The vectorized variant keeps four float32 (16 B) per thread.
VEC_BLOCK, VEC_WARPS = 1024, 8
VEC_THREADS = 32 * VEC_WARPS
vec_per_thread = VEC_BLOCK // VEC_THREADS
assert (VEC_THREADS, vec_per_thread) == (256, 4)
assert vec_per_thread * 4 == 16

# Register-file arithmetic for an illustrative SM with 64K 32-bit registers.
regs_per_sm = 65536
regs_per_thread = 64
threads_by_regs = regs_per_sm // regs_per_thread
assert threads_by_regs == 1024
assert threads_by_regs // 32 == 32

# Shared-memory staging for a triple-buffered 64x64x64 float16 GEMM tile.
BM = BN = BK = 64
b = 2
stages = 3
panel_bytes = (BM * BK + BK * BN) * b
smem_bytes = stages * panel_bytes
assert panel_bytes == 16384
assert smem_bytes == 49152
assert smem_bytes == 48 * 1024
assert (100 * 1024) // smem_bytes == 2

# Register blocking: a 4x64 tile loads its row vector once.
RB_M, RB_N = 4, 64
assert RB_M * RB_N == 256
assert (RB_M * RB_N) // RB_N == RB_M == 4

print("resources: 8 values/thread, 16 for two tiles, 48 KiB smem, 4x row reuse")


def test_resource_budget():
    assert (BLOCK // T, VEC_BLOCK // VEC_THREADS) == (8, 4)
    assert smem_bytes == 48 * 1024
    assert (RB_M * RB_N) // RB_N == 4

### 13.10 Access patterns

**Connection.** Builds on 13.9: resource budgets are irrelevant if the
lane-address map scatters requests.

**Rule.** Coalescing is a property of the lane map:
$\Delta a=a_{\ell+1}-a_\ell=b$.

**Formula.** A warp touches $\lceil 32\Delta a/L\rceil$ lines; the table
evaluates the common maps.

**Example.** For a $32\times32$ `float32` matrix:

| pattern | lane address | lines/warp |
|---|---|---:|
| contiguous 1-D | $a_0+4\ell$ | 1 |
| row of row-major | $a_0+4\ell$ | 1 |
| column of $32\times32$ | $a_0+128\ell$ | 32 |
| stride 4 (`float32`) | $a_0+16\ell$ | 4 |
| stride 32 (`float32`) | $a_0+128\ell$ | 32 |

A column read is $32\times$ worse than a row read.

**Where.** Loop order, tile orientation, transposes; the pattern decides
whether a tile is a line, a sector, or a scattering of lines.

**Triton.** Make the fastest-varying logical axis the lane axis; load a 2-D tile
and change the logical layout before the store.

**Failure.** Using `tl.arange` to index a column, or letting the outer loop vary
fastest inside a program.

**Bridge.** With the map fixed, the remaining waste is repeated work.

### 13.11 Avoid redundant loads

**Connection.** Builds on 13.10: the lane map is now good, so the next waste is
reading the same bytes twice.

**Rule.** Load each byte once per program and keep it in a register.

**Formula.** A $k$-op chain that materializes intermediates moves $Q_k=2kNb$;
fusion gives $Q=2Nb$. For $k=3$, $6Nb\to2Nb$, saving $4Nb$; at $N=2^{20}$ and
`float32`, that is $16$ MiB.

**Example.** `x*x + 2*x + 1` with three separate `tl.load(x_ptr + offs)` calls
issues three reads; one named tile issues one.

**Where.** Multi-term expressions, normalization, epilogues; HBM is two orders
of magnitude slower than a register.

**Triton.** Compute `alpha * x + beta` between one `tl.load` and one `tl.store`;
reuse a named tile.

**Failure.** A two-kernel chain (`scale` then `shift`) reloads the intermediate;
repeated `tl.load` calls in one expression reread the same line.

**Bridge.** The same argument applies to the write side.

### 13.12 Avoid redundant stores

**Connection.** Builds on 13.11: after eliminating repeated reads, eliminate
repeated writes.

**Rule.** Do not write a value that the next operation immediately reads.

**Formula.** An intermediate costs a store $Nb$ plus a later load $Nb$; for
$t=\alpha x$, $y=t+\beta$, $Q_{\text{two}}=4Nb$ and $Q_{\text{fused}}=2Nb$, a
$2Nb$ saving (half the traffic); at $N=2^{26}$ and `float32`, $512$ MiB.

**Example.** The two-pass form reads $x$, writes $t$, reads $t$, writes $y$;
the fused form reads $x$ and writes $y$ once.

**Where.** Epilogues, activation chains, bias adds; stores consume bandwidth,
can allocate a line, and usually force a kernel boundary.

**Triton.** Fuse the epilogue; keep the intermediate tile live; cast to the
output dtype only at the final store.

**Failure.** "Store early to free registers" adds HBM traffic when the value is
consumed immediately; store only for a real output or a genuine spill.

**Where this leads.** The worked example fixes the $N=1024$ arithmetic; Method
01 makes the strided/contiguous address map explicit; Method 02 builds
coalesced and strided kernels; Method 03 fuses, vectorizes, and register-blocks
the affine chain.

## Chapter map

| § | Topic | Rule/formula | Traffic implication | Kernel/demo |
|---|---|---|---|---|
| **13.1** | Coalesced access | $T_{\text{line}}=\lceil 32b/128\rceil$ | one line per warp for `float32` | `scale_coalesced` |
| **13.2** | Uncoalesced access | $T_{\text{line}}=\lceil 32sb/128\rceil$ | up to $s\times$ overfetch | `scale_strided` |
| **13.3** | Memory alignment | $a\bmod w=0$ | misalignment splits vectors | `fused_affine_vec` |
| **13.4** | Vectorized loads | $v=128/b$ | $B/v$ 16-byte loads | `fused_affine_vec` |
| **13.5** | Vectorized stores | $B/v$ 16-byte stores | same line count, fewer instructions | `fused_affine_vec` |
| **13.6** | Memory bandwidth | $\mathrm{BW}=Q/t$ | roofline $t\ge Q/\mathrm{BW}$ | performance model |
| **13.7** | Cache behavior | $W\le C_{\text{L2}}$ | reuse cuts HBM bytes | `row_broadcast` |
| **13.8** | Register usage | $e=B/(32W)$ | spills add local-memory traffic | `fused_affine_vec` |
| **13.9** | Shared memory | $s(B_MB_K+B_KB_N)b$ | staging is an occupancy budget | resource arithmetic |
| **13.10** | Access patterns | $\Delta a=b$ across lanes | row 1 line, column 32 lines | `row_broadcast` |
| **13.11** | Avoid redundant loads | $2kNb\to2Nb$ | one read per byte | `fused_affine` |
| **13.12** | Avoid redundant stores | $4Nb\to2Nb$ | no intermediate HBM | `fused_affine` |
| **★** | Capstone | all of the above | $2\times$ traffic bound from fusion | fused + vectorized kernel |

## Worked numerical example: a 1-D copy of $N=1024$ `float32`

**Connection.** Applies the Part I rules to one tile: $N=1024$, $b=4$ B,
$L=128$ B, $B=256$.

**Traffic.** $Q_r=Nb=4096$ B, $Q_w=4096$ B, $Q=8192$ B $=8$ KiB.

**Lines.** $L/b=32$ elements per line, so each direction touches
$4096/128=32$ lines and the copy touches $64$.

**Tiles.** $P=\lceil N/B\rceil=1024/256=4$ programs; each tile is
$256\cdot4/128=8$ lines.

**Threads and vectors.** $v=128/32=4$ `float32` per $16$-byte transaction, and
$e=B/T$:

| threads $T$ | warps $W$ | elements/thread $e$ | per-thread access |
|---:|---:|---:|---|
| 32 | 1 | 8 | two 16-byte vectors |
| 64 | 2 | 4 | one 16-byte vector |
| 128 | 4 | 2 | one 8-byte vector |
| 256 | 8 | 1 | one 4-byte scalar |

- Four elements per thread means $T=64$ (two warps), not $T=128$: a $4$-warp
  program covers $256$ elements with two elements per thread, an $8$-byte
  access.
- A $256$-element tile at four elements per thread needs $256/4=64$ threads;
  four warps at four elements per thread cover $128\cdot4=512$ elements.

**Contiguous versus stride-4.** A stride-4 tile touches
$0,16,\ldots,4080$ B. Its span is $256\cdot4\cdot4=4096$ B, or $32$ lines,
against $8$ contiguous lines: $4\times$ the traffic. Per warp: $4$ lines at
$s=4$, $32$ lines at $s=32$.

**Bandwidth.** At $t=4.0\ \mu s$,
$\mathrm{BW}=8192/(4.0\times10^{-6})=2.048\times10^9$ B/s $=2.05$ GB/s. The
$3$ TB/s roof is $8192/(3\times10^{12})=2.73$ ns, so this is launch overhead.
The performance model uses $N=2^{26}$, where $Q=512$ MiB and
$t=(512\ \text{MiB})/(3.0\ \text{TB/s})=0.179$ ms.

**Alignment.** With $a_{\text{base}}=4096$ and $B=256$, tile starts
$4096+1024p$ are divisible by $16$ and $128$; $+12$ B is not $16$-byte aligned
and must split.

In [ ]:
# Why: verify every byte and cache-line number in the worked example.
N, b, L = 1024, 4, 128
B, stride = 256, 4

q_read = N * b
q_write = N * b
q_total = q_read + q_write
assert (q_read, q_write, q_total) == (4096, 4096, 8192)

per_line = L // b
assert per_line == 32
assert (32 * 1 * b) // L == 1
read_lines = q_read // L
write_lines = q_write // L
assert (read_lines, write_lines, read_lines + write_lines) == (32, 32, 64)

P = (N + B - 1) // B
assert P == 4
tile_bytes = B * b
tile_lines = tile_bytes // L
assert (tile_bytes, tile_lines) == (1024, 8)
assert P * tile_lines == 32

mapping = {32: (1, 8), 64: (2, 4), 128: (4, 2), 256: (8, 1)}
for threads, (warps, per_thread) in mapping.items():
    assert threads // 32 == warps
    assert B // threads == per_thread
assert B // 64 == 4          # four float32 per thread -> one 16-byte vector
assert B // 32 == 8          # one warp owns eight float32 -> two vectors
assert 4 * 32 * 4 == 512     # four warps at four elements each cover 512

v = 128 // (8 * b)
assert v == 4
assert B // v == 64

span = B * stride * b
stride_lines = span // L
assert span == 4096 and stride_lines == 32
assert stride_lines // tile_lines == stride == 4

warp_span_stride4 = 32 * stride * b
assert warp_span_stride4 == 512
assert warp_span_stride4 // L == 4

warp_span_stride32 = 32 * 32 * b
assert warp_span_stride32 == 4096
assert warp_span_stride32 // L == 32
assert 32 // 1 == 32

base = 4096
assert base % L == 0
assert (base + 3 * b) % 16 == 12
assert (base + 4 * B) % 16 == 0
assert (base + 1024 * 3) % 16 == 0

t_tiny = 4.0e-6
bw_tiny = q_total / t_tiny
assert abs(bw_tiny - 2.048e9) < 1.0
t_roof_tiny = q_total / 3.0e12
assert abs(t_roof_tiny - 2.7306666666666667e-9) < 1e-21

print("N=1024: 8 KiB, 64 lines; tile 256: 8 lines; stride 4: 32 lines (4x)")


def test_worked_arithmetic():
    assert (q_total, read_lines + write_lines, P, tile_lines) == (8192, 64, 4, 8)
    assert stride_lines == 4 * tile_lines
    assert mapping[64] == (2, 4)

## Part II — Method 01: naive strided and contiguous references

**Connection.** Chapter 12's lane map plus §13.1–13.2; the reference makes the
address map visible without a GPU.

- **Reference.** A Python loop is the correctness oracle and the address-map
  model; it is intentionally too slow for a device workload.
- **Row-major address.** For $X\in\mathbb{R}^{8\times16}$,
  $a(r,c)=a_0+4(16r+c)$.
- **Contiguous loop.** $c\mapsto c+1$, so $\Delta a=4$ B; a warp covers
  $128$ B, or one line.
- **Column loop.** $r\mapsto r+1$, so $\Delta a=16\cdot4=64$ B; a warp spans
  $32\cdot64=2048$ B, or $16$ lines.
- **Transpose.** `xt = x.t()` has stride $(1,16)$; a row of `xt` is a column of
  `x`, so walking it row-wise is strided.
- **Cost.** Both loops perform the same $128$ multiplies; the column order
  reads $16\times$ the lines.
- **No timing.** A CPU timing would not transfer to the device; the line count
  is the model.

In [ ]:
# Why: the same values with two address orders; strides explain the speed gap.
def naive_contiguous_scale(x, alpha):
    out = torch.empty_like(x)
    xf, of = x.reshape(-1), out.reshape(-1)
    for i in range(xf.numel()):
        of[i] = alpha * xf[i]
    return out


def naive_column_scale(x, alpha):
    rows, cols = x.shape
    out = torch.empty_like(x)
    for c in range(cols):
        for r in range(rows):
            out[r, c] = alpha * x[r, c]
    return out


def test_method01_reference():
    rows, cols = 8, 16
    x = torch.arange(rows * cols, dtype=torch.float32).view(rows, cols)
    alpha = 1.5
    assert x.shape == (rows, cols)
    assert x.stride() == (cols, 1)
    assert x[0].stride() == (1,)
    assert x[:, 0].stride() == (cols,)
    assert not x[:, 0].is_contiguous()
    assert x.t().stride() == (1, cols)

    assert torch.equal(naive_contiguous_scale(x, alpha), alpha * x)
    assert torch.equal(naive_column_scale(x, alpha), alpha * x)

    column_stride = cols
    warp_span = 32 * column_stride * 4
    lines = warp_span // 128
    assert column_stride == 16 and warp_span == 2048 and lines == 16
    print("method 01: row stride 1 (1 line/warp), column stride 16 (16 lines/warp)")


test_method01_reference()

## Part III — Method 02: coalesced and strided Triton kernels

**Connection.** Applies §13.1–13.2 to Triton; both kernels use
$\text{offs}_p=Bp+\{0,\ldots,B-1\}$, mask $\text{offs}_p\lt N$, and
$P=\lceil N/B\rceil$.

**Tile map.** For $N=1024$ and $B=256$:

| `pid` | offsets | lines | mask |
|---:|---|---:|---|
| 0 | 0 … 255 | 0 … 7 | all true |
| 1 | 256 … 511 | 8 … 15 | all true |
| 2 | 512 … 767 | 16 … 23 | all true |
| 3 | 768 … 1023 | 24 … 31 | all true |

- **Coverage.** Four tiles cover all $32$ lines of the vector; a tail tile has
  the same line count but a mask that removes lanes.
- **Coalesced kernel.** One lane per element; the load and the store are both
  unit-stride.
- **Strided kernel.** Compacts a stride-$s$ slice:
  $\text{src}_\ell=s(Bp+\ell)$, $\text{dst}_\ell=Bp+\ell$.
- **Trade.** The load is uncoalesced; the store is coalesced; compare with
  `x[::s]` on the host.
- **Bridge.** Method 03 removes the separate pass these kernels still pay for.

In [ ]:
@triton.jit
def scale_coalesced_kernel(x_ptr, out_ptr, n, alpha, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, alpha * x, mask=mask)


def scale_coalesced(x, alpha=1.0, BLOCK=256):
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n:
        grid = (triton.cdiv(n, BLOCK),)
        scale_coalesced_kernel[grid](x, out, n, float(alpha), BLOCK=BLOCK)
    return out

In [ ]:
def test_coalesced_scale():
    require_gpu("coalesced scale")
    torch.manual_seed(0)
    for n in (1000, 1024):
        x = torch.randn(n, device=DEVICE)
        torch.testing.assert_close(scale_coalesced(x, 2.0), 2.0 * x)
    print("method 02: coalesced scale OK (full tile and masked tail)")


run_if_gpu(test_coalesced_scale)

In [ ]:
@triton.jit
def scale_strided_kernel(x_ptr, out_ptr, n, stride, n_out, alpha, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    j = pid * BLOCK + tl.arange(0, BLOCK)
    src = j * stride
    mask = (j < n_out) & (src < n)
    x = tl.load(x_ptr + src, mask=mask, other=0.0)
    tl.store(out_ptr + j, alpha * x, mask=mask)


def scale_strided(x, stride=4, alpha=1.0, BLOCK=256):
    x = x.contiguous()
    n = x.numel()
    n_out = (n + stride - 1) // stride
    out = torch.empty(n_out, dtype=x.dtype, device=x.device)
    if n_out:
        grid = (triton.cdiv(n_out, BLOCK),)
        scale_strided_kernel[grid](x, out, n, stride, n_out, float(alpha), BLOCK=BLOCK)
    return out

In [ ]:
def test_strided_scale():
    require_gpu("strided scale")
    torch.manual_seed(0)
    x = torch.randn(1024, device=DEVICE)
    torch.testing.assert_close(scale_strided(x, 4, 2.0), x[::4] * 2.0)
    head = x[:1000]
    torch.testing.assert_close(scale_strided(head, 4, 2.0), head[::4] * 2.0)
    print("method 02: stride-4 compaction OK (loads strided, stores contiguous)")


run_if_gpu(test_strided_scale)

## Part IV — Method 03: fuse, vectorize, and block

**Connection.** Builds on §13.11–13.12; the affine chain is the smallest
example with a redundant pass.

- **Chain.** $t=\alpha x$, $y=t+\beta$ becomes $y=\alpha x+\beta$.
- **Traffic.** $Q_{\text{two}}=4Nb=16N$ bytes and $Q_{\text{fused}}=2Nb=8N$
  bytes; fusion halves traffic and saves $512$ MiB at $N=2^{26}$ `float32`.
- **Vectorized variant.** `BLOCK=1024`, `num_warps=8` gives $T=256$ and
  $e=4$ `float32` per thread, one $16$-byte vector. Hint:
  `offs = tl.max_contiguous(tl.multiple_of(offs, 4), 4)`.
- **Register blocking.** A 2-D kernel loads a row tile of $B_N$ values once and
  reuses it across $B_M$ rows; row loads fall from $B_MB_N$ to $B_N$, a factor
  $B_M$.
- **Bridge.** The budget arithmetic for both levels is in §13.8–13.9.

In [ ]:
@triton.jit
def scale_kernel(x_ptr, out_ptr, n, alpha, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, alpha * x, mask=mask)


@triton.jit
def shift_kernel(x_ptr, out_ptr, n, beta, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, x + beta, mask=mask)


@triton.jit
def fused_affine_kernel(x_ptr, out_ptr, n, alpha, beta, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, alpha * x + beta, mask=mask)


def two_pass_affine(x, alpha, beta, BLOCK=256):
    x = x.contiguous()
    tmp = torch.empty_like(x)
    out = torch.empty_like(x)
    n = x.numel()
    if n:
        grid = (triton.cdiv(n, BLOCK),)
        scale_kernel[grid](x, tmp, n, float(alpha), BLOCK=BLOCK)
        shift_kernel[grid](tmp, out, n, float(beta), BLOCK=BLOCK)
    return out


def fused_affine(x, alpha, beta, BLOCK=256):
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n:
        grid = (triton.cdiv(n, BLOCK),)
        fused_affine_kernel[grid](x, out, n, float(alpha), float(beta), BLOCK=BLOCK)
    return out

In [ ]:
@triton.jit
def fused_affine_vec_kernel(x_ptr, out_ptr, n, alpha, beta, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    offs = tl.max_contiguous(tl.multiple_of(offs, 4), 4)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask, other=0.0)
    tl.store(out_ptr + offs, alpha * x + beta, mask=mask)


def fused_affine_vec(x, alpha, beta, BLOCK=1024):
    x = x.contiguous()
    out = torch.empty_like(x)
    n = x.numel()
    if n:
        grid = (triton.cdiv(n, BLOCK),)
        fused_affine_vec_kernel[grid](
            x, out, n, float(alpha), float(beta), BLOCK=BLOCK, num_warps=8
        )
    return out

In [ ]:
# Why: verify the fusion arithmetic on the CPU before trusting a GPU kernel.
def test_method03_cpu_reference():
    x = torch.arange(12, dtype=torch.float32)
    alpha, beta = 2.0, 1.0
    tmp = alpha * x
    two = tmp + beta
    fused = alpha * x + beta
    assert torch.equal(two, fused)
    assert fused.tolist() == [1.0, 3.0, 5.0, 7.0, 9.0, 11.0,
                              13.0, 15.0, 17.0, 19.0, 21.0, 23.0]

    n, b = 1 << 26, 4
    q_two = 4 * n * b
    q_fused = 2 * n * b
    assert q_two == 2 * q_fused
    assert (q_two, q_fused) == (16 * n, 8 * n)
    assert q_two - q_fused == 512 * 1024 * 1024
    print("method 03 CPU: fusion halves traffic (512 MiB saved at N=2^26)")


test_method03_cpu_reference()

In [ ]:
def test_method03_triton():
    require_gpu("method 03")
    torch.manual_seed(0)
    for n in (1000, 1024):
        x = torch.randn(n, device=DEVICE)
        ref = 2.0 * x + 1.0
        torch.testing.assert_close(two_pass_affine(x, 2.0, 1.0), ref)
        torch.testing.assert_close(fused_affine(x, 2.0, 1.0), ref)
        torch.testing.assert_close(fused_affine_vec(x, 2.0, 1.0), ref)
    print("method 03: two-pass, fused, and 4-elements/thread variants agree")


run_if_gpu(test_method03_triton)

### Register blocking and shared memory in one picture

**Connection.** Ties §13.8–13.9 to Method 03's 2-D kernel.

- **Register tile.** `row = tl.load(row_ptr + cols, ...)` puts $B_N$ values in
  registers.
- **Reuse.** `row[None, :]` broadcasts across $B_M$ rows; row loads are $B_N$
  instead of $B_MB_N$, a factor $B_M$.
- **Price.** The row tile stays live while the $B_M\times B_N$ element tile is
  processed, which is the §13.8 register-pressure trade-off.
- **Shared memory.** `tl.dot` and layout conversions stage panels with
  $M_{\text{smem}}=s(B_MB_K+B_KB_N)b$; the resource cell evaluates $48$ KiB
  for a triple-buffered $64^3$ `float16` tile.
- **Lesson.** Reuse saves HBM traffic, but the staging buffer limits resident
  programs.

In [ ]:
@triton.jit
def row_broadcast_kernel(x_ptr, row_ptr, out_ptr, rows, cols,
                         BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    rm = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    rn = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask = (rm[:, None] < rows) & (rn[None, :] < cols)
    x = tl.load(x_ptr + rm[:, None] * cols + rn[None, :], mask=mask, other=0.0)
    row = tl.load(row_ptr + rn, mask=rn < cols, other=0.0)
    tl.store(out_ptr + rm[:, None] * cols + rn[None, :], x + row[None, :], mask=mask)


def row_broadcast(x, row, BLOCK_M=4, BLOCK_N=8):
    x = x.contiguous()
    row = row.contiguous()
    out = torch.empty_like(x)
    rows, cols = x.shape
    grid = (triton.cdiv(rows, BLOCK_M), triton.cdiv(cols, BLOCK_N))
    row_broadcast_kernel[grid](x, row, out, rows, cols,
                               BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N)
    return out

In [ ]:
def test_row_broadcast():
    require_gpu("register-blocked row broadcast")
    torch.manual_seed(0)
    for rows, cols in ((7, 10), (33, 17)):
        x = torch.randn(rows, cols, device=DEVICE)
        row = torch.randn(cols, device=DEVICE)
        torch.testing.assert_close(row_broadcast(x, row), x + row)
    print("method 03: register-blocked row broadcast OK (row loaded once per tile)")


run_if_gpu(test_row_broadcast)

## Part V — Performance model

**Connection.** Applies §13.6–13.7 to the fused and two-pass affine kernels; no
hardware timing is required.

- **Size.** $N=2^{26}=67{,}108{,}864$ `float32`; one vector is
  $Nb=256$ MiB.
- **Copy traffic.** $Q_{\text{copy}}=2Nb=536{,}870{,}912$ B $=512$ MiB.
- **Copy roof.** At $\mathrm{BW}=3.0$ TB/s,
  $t_{\text{copy}}=Q/\mathrm{BW}=1.7896\times10^{-4}$ s $=0.179$ ms.
- **Two-pass versus fused.** $Q_{\text{two}}=4Nb=1$ GiB gives $0.358$ ms;
  $Q_{\text{fused}}=2Nb=512$ MiB gives $0.179$ ms; the bound is
  $S=Q_{\text{two}}/Q_{\text{fused}}=2$.
- **Intensity.** $y=\alpha x+\beta$ costs $2$ FLOPs and $8$ bytes per element,
  so $\mathrm{AI}=2N/(8N)=0.25$ FLOP/B; a machine balance of $10$–$100$
  FLOP/B makes it memory-bound.
- **Compute roof.** At $60$ TFLOP/s,
  $t_{\text{compute}}=2N/(60\times10^{12})=2.24\ \mu s\ll0.179$ ms.
- **Achieved.** If a measured copy takes $0.20$ ms, $Q/t=2.68$ TB/s, about
  $89\%$ of $3.0$ TB/s.
- **Cache caveat.** A $1024\times1024$ `float32` matrix is $4$ MiB and
  $32{,}768$ lines; it fits a large L2, so a second pass can measure above the
  DRAM roof.

In [ ]:
# Why: every roofline number in Part V is arithmetic, so it can be asserted.
def test_performance_model():
    N, b = 1 << 26, 4
    q_copy = 2 * N * b
    q_two = 4 * N * b
    q_fused = 2 * N * b
    assert q_copy == q_fused == 536_870_912 == 512 * 1024 ** 2
    assert q_two == 1_073_741_824 == 1024 ** 3
    assert q_two - q_fused == 536_870_912

    BW = 3.0e12
    t_copy = q_copy / BW
    t_two = q_two / BW
    assert abs(t_copy - 1.7895697066666668e-4) < 1e-16
    assert abs(t_two - 3.5791394133333336e-4) < 1e-16
    assert abs(t_two / t_copy - 2.0) < 1e-12

    measured = 2.0e-4
    achieved = q_copy / measured
    assert abs(achieved - 2.68435456e12) < 1.0
    assert abs(achieved / BW - 0.8947848533333333) < 1e-12

    flops = 2 * N
    ai = flops / q_fused
    assert abs(ai - 0.25) < 1e-12
    peak = 60.0e12
    t_compute = flops / peak
    assert abs(t_compute - 2.2369621333333333e-6) < 1e-16
    assert t_compute < t_copy / 10

    n_small, b_small = 1 << 20, 4
    q_chain = 6 * n_small * b_small
    q_one = 2 * n_small * b_small
    assert q_chain - q_one == 16 * 1024 * 1024
    assert q_chain // q_one == 3

    matrix_bytes = 1024 * 1024 * b_small
    assert matrix_bytes == 4 * 1024 * 1024
    assert matrix_bytes // 128 == 32768

    print("roofline: 512 MiB, 0.179 ms at 3 TB/s, AI 0.25; fusion bound 2x")


test_performance_model()

## Pitfalls

| Pitfall | What happens | Fix |
|---|---|---|
| Misaligned base | a $16$-byte vector splits into narrower accesses | align the base and use power-of-two tiles; verify `ptr % 16` |
| Stride not 1 | up to $s\times$ more lines per warp | make the lane axis unit-stride; transpose the logical tile |
| Too few elements per thread | scalar accesses, issue-bound instead of bandwidth-bound | choose `BLOCK` so $e=B/(32W)\ge4$ for `float32` |
| Too many registers | spills to local memory, extra HBM traffic | smaller `BLOCK`, fewer live tiles, split the kernel |
| Redundant loads | $k$ passes read the same bytes $k$ times | fuse and reuse a register tile |
| Writing through a cache | a partial store allocates a line and costs a read | write the final output once, in vector order |
| Assuming shared memory is free | occupancy drops; bank conflicts appear | budget $s(B_MB_K+B_KB_N)b$ bytes per block |
| Timing one small launch | measures launch overhead, not bandwidth | use large $N$, many repetitions, and report $Q/t$ |

The through-line: every row is either a line-count mistake, a resource-budget
mistake, or a traffic mistake. The bytes and the budgets are checkable on paper
before a profiler is opened.

## Self-check

1. For $N=1024$ `float32`, how many bytes and how many $128$-byte lines does a
   copy move? How many lines does one $256$-element tile touch?
2. A warp reads 32 `float32` values at stride $4$. How many lines does it
   touch, and what fraction of each line is useful?
3. Why is four elements per thread a $16$-byte vector, and how many threads does
   a $256$-element tile need to achieve it?
4. A base pointer is $4$-byte aligned but not $16$-byte aligned. What happens to
   a would-be `v4.b32` load?
5. State the roofline bound and compute the expected time for $Q=512$ MiB at
   $3$ TB/s.
6. Why can a measured bandwidth exceed the DRAM specification? Give a concrete
   working-set size from this chapter.
7. Derive the per-thread register estimate for $B=1024$, $W=4$, and two live
   tiles. What changes if you halve `BLOCK`?
8. How many bytes does a two-pass affine chain move per element, and how many
   does the fused form move? Where does the difference go?
9. Why is a column read of a row-major $32\times32$ matrix $32\times$ worse than
   a row read?
10. A kernel has $Q=8N$ bytes and $\mathrm{AI}=0.25$ FLOP/B. Is it memory- or
    compute-bound on a machine with balance $10$ FLOP/B? What if the balance
    were $0.1$ FLOP/B?

## Summary

Memory optimization is the discipline of counting bytes and lines before
counting instructions.

- **Coalescing** turns 32 lane addresses into one line (`float32`) or four lines
  at stride $4$. The line count, not the useful-byte count, is the traffic.
- **Alignment** makes vector instructions legal; a $16$-byte access needs
  $a\bmod16=0$, and a power-of-two tile keeps every tile start aligned.
- **Vectorization** moves $v=128/b$ elements per instruction. Four `float32` per
  thread is a $16$-byte vector; the thread count, not the warp count, sets it.
- **Bandwidth** is $Q/t$; the roofline is $t\ge Q/\mathrm{BW}$, and arithmetic
  intensity decides which roof applies.
- **Cache** reuse can beat the DRAM roof when the working set fits; a $4$ MiB
  matrix fits a large L2.
- **Registers** hold the tile between operations; too many live tiles spill to
  HBM-backed local memory.
- **Shared memory** stages reusable panels and is a hard occupancy budget:
  $48$ KiB for a triple-buffered $64^3$ `float16` tile.
- **Access patterns** are the lane-address map. A row is one line; a column of a
  $32\times32$ matrix is $32$ lines.
- **Redundant loads and stores** are the easiest traffic to remove: fuse the
  chain, keep the intermediate in a register, and write the final value once.

The fused affine kernel is the smallest complete example: one load, one
multiply-add, one store, four elements per thread, and a $2\times$ traffic bound
over the two-pass version.

**Where this leads.** The exercises extend the line-count, vectorization,
roofline, register, and fusion arguments; Chapter 14 reuses the same traffic
model for reductions.

[Index](../../00_index.ipynb) · [← Previous](../ch12_layouts/12_complete.ipynb) · [Next →](../../part6_reductions/ch14_reductions/14_complete.ipynb)

## Exercises

1. **Line count by hand.** For $N=4096$ `float32` and stride $8$, compute the
   contiguous and strided line counts. Verify with a small Python function that
   counts distinct `addr // 128` values for one warp.
2. **Tail vectorization.** Run the coalesced scale kernel with
   $N=1025$ and `BLOCK=256`. Which tile loses vectorization, and how many scalar
   accesses remain? Explain using the mask.
3. **Alignment experiment.** Pass `x[1:]` to the coalesced kernel and print
   `data_ptr() % 16`. Does the compiler still emit a vector load? Remove the
   alignment hint and compare correctness.
4. **Stride sweep.** Time or count lines for $s=1,2,4,8,16,32$ on a $1$-D array
   of $2^{20}$ `float32`. Plot the ratio against the formula
   $32sb/128$ and explain the plateaus.
5. **Roofline.** For $N=2^{24}$ and $b=4$, compute $Q$, $\mathrm{AI}$, and the
   expected time at $2$ TB/s for a copy, a fused affine, and a two-pass affine.
   Which pair has a $2\times$ bound?
6. **Fusion chain.** Fuse `y = 3*x*x + 2*x + 1` into one kernel. Count the loads
   and stores per element; compare with a three-kernel version. Check the result
   against PyTorch.
7. **Register pressure.** Take the fused kernel and deliberately keep three
   large live tiles. Compile with two `BLOCK` values and inspect the register and
   spill metadata. Which one spills?
8. **Shared-memory budget.** For `float16` and $B_M=B_N=128$, $B_K=32$, compute
   the staged bytes for $s=2,3,4$. How many blocks fit in $100$ KiB, and what
   happens to occupancy?
9. **Column kernel.** Write a kernel that reads a $32\times32$ `float32` column
   and writes it contiguously. Compute the line count before and after the
   transpose. Verify with `x.t().contiguous()`.
10. **Cache-aware sweep.** Read a $4096\times4096$ `float32` matrix row-wise, then
    column-wise, and compute the ratio of line touches. Why does the measured
    ratio depend on whether the matrix fits in L2?

In [ ]:
CPU_TESTS = [
    test_worked_arithmetic,
    test_method01_reference,
    test_method03_cpu_reference,
    test_resource_budget,
    test_performance_model,
]

GPU_TESTS = [
    test_coalesced_scale,
    test_strided_scale,
    test_method03_triton,
    test_row_broadcast,
]

print("CPU tests")
for fn in CPU_TESTS:
    fn()

print("\nGPU tests (skipped without CUDA/ROCm)")
for fn in GPU_TESTS:
    run_if_gpu(fn)

print("\ndone: CPU checks passed; GPU checks ran or printed SKIP")